# LaughTuned — Demo Notebook

Fine-tuning Mistral-7B-Instruct-v0.2 for comedy writing using **DPO** and **KTO**, both implemented from scratch in PyTorch (CS 5788, Cornell).

This notebook demonstrates the pipeline. The main engine lives in the `.py` modules of the repo.

## Step 0 — Environment Setup

Bootstraps the Colab runtime: clones the code repo, installs dependencies, mounts Drive, sets seeds, and verifies the GPU.

In [ ]:
# === Colab bootstrap: clone repo, install deps, cd into the code dir ===
# Edit REPO_URL below to point at your GitHub fork before running on Colab.
import os
import subprocess
import sys

REPO_URL = "https://github.com/pcatattacks/laughtuned.git" 
REPO_DIR = "/content/laughtuned"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        print(f"Cloning {REPO_URL} into {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        print(f"Updating existing checkout at {REPO_DIR} ...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    os.chdir(REPO_DIR)
    print(f"cwd: {os.getcwd()}")
    print("Installing dependencies (quiet) ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )
else:
    print("Not on Colab — assuming dependencies are already installed and cwd is the repo root.")

In [ ]:
# === Mount Drive and create the artifact tree ===
from config import CONFIG
from utils.drive_utils import mount_drive, ensure_drive_dirs

mount_drive()
ensure_drive_dirs(CONFIG)

In [ ]:
# === Set all random seeds for reproducibility ===
import random
import numpy as np
import torch

SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Seeded with {SEED}")

In [ ]:
# === GPU sanity check ===
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | total VRAM: {total_gb:.1f} GB")
else:
    print("No GPU detected. Switch the Colab runtime to T4 or A100 before continuing.")

## Step 1 — Load the base model with QLoRA

`load_model_and_tokenizer` does the QLoRA setup: 4-bit NF4 with double quantization, then LoRA adapters (rank 16) on the attention projections. Expect a ~4 GB base load; trainable params should be well under 1% of total.

In [ ]:
from models.load_model import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(CONFIG)

## `compute_log_probs` — the shared primitive for DPO and KTO

Both losses score a response by the total log-probability the policy assigns to it given the prompt: `log π(y | x) = Σ_t log π(y_t | x, y_<t)`. The implementation in [`models/log_probs.py`](models/log_probs.py) handles three subtleties:

1. **Shift by one.** A causal LM's output at position `t` predicts the token at `t+1`, so we align `logits[:, :-1]` with `input_ids[:, 1:]` and `label_mask[:, 1:]`.
2. **Prompt tokens contribute zero.** The label mask is 1 only on response tokens (and 0 on prompt tokens *and* padding); after shifting we multiply by it before summing.
3. **Sum, not mean.** DPO and KTO are derived from the total sequence log-probability; length-normalization changes the optimization landscape.

The smoke test below verifies shape, sign, and masking on a synthetic batch.

In [ ]:
# === Smoke test for compute_log_probs ===
from models.log_probs import compute_log_probs

# Synthetic batch: 2 examples, 8 tokens each.
# First 4 tokens are "prompt" (mask=0), last 4 are "response" (mask=1).
B, T = 2, 8
device = next(model.parameters()).device
vocab_size = model.config.vocab_size

input_ids_B_T = torch.randint(0, vocab_size, (B, T), device=device)
attention_mask_B_T = torch.ones(B, T, dtype=torch.long, device=device)
label_mask_B_T = torch.zeros(B, T, dtype=torch.long, device=device)
label_mask_B_T[:, T // 2 :] = 1  # response = second half

with torch.no_grad():
    log_probs_B = compute_log_probs(
        model, input_ids_B_T, attention_mask_B_T, label_mask_B_T
    )

# Test 1: shape
assert log_probs_B.shape == (B,), f"expected ({B},), got {tuple(log_probs_B.shape)}"

# Test 2: all values <= 0
assert (log_probs_B <= 0).all(), f"log-probs should be non-positive, got {log_probs_B}"

# Test 3: zeroed mask -> zero output
zero_mask_B_T = torch.zeros_like(label_mask_B_T)
with torch.no_grad():
    zero_log_probs_B = compute_log_probs(
        model, input_ids_B_T, attention_mask_B_T, zero_mask_B_T
    )
assert torch.allclose(
    zero_log_probs_B, torch.zeros_like(zero_log_probs_B)
), f"expected all-zero output for empty mask, got {zero_log_probs_B}"

print("compute_log_probs passed all 3 smoke tests.")
print(f"sample log-probs: {log_probs_B.tolist()}")